In [ ]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GITHUB_TOKEN"] = user_secrets.get_secret("GITHUB_HSE_HW")
os.environ["COMET_API_KEY"] = user_secrets.get_secret("COMET_API_KEY")

In [ ]:
import shutil
import subprocess

if shutil.which("gh") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gh"], check=True)

%cd /kaggle/working
!gh repo clone https://github.com/Yaitco/NLP-HW2.git
%cd NLP-HW2

Включи Internet и GPU. В Secrets разреши этому ноутбуку доступ к `GITHUB_HSE_HW` и `COMET_API_KEY`.

Через **Add Input** подключи свой Kaggle Dataset с подготовленными файлами из локального репозитория:

```text
<INPUT_ROOT>/
    model_best.pth
    artifacts/
        tokenizer_noimports_16k.json
        length128_noimports_16k/
            train_indices.npy
            train_indices.json
        tokenized_noimports_full_clean_16k/
            dataset_dict.json
            preparation.json
            train/       # Все файлы, включая .arrow
            validation/
            test/
```

Возьми checkpoint из нужного завершённого этапа обучения вместе с соответствующими ему данными и токенизатором. Пути в `artifacts` должны совпадать с конфигом этого checkpoint. Меняй `INPUT_ROOT` ниже на реальный путь из панели Input.

Данные читаются по символическим ссылкам из Input; поиск, новые веса и логи пишутся в `/kaggle/working/NLP-HW2`. GPU используется одна. Валидация исполняет код в отдельных процессах с лимитами времени/памяти, без Bubblewrap (`sandbox=False`). Секреты не выводятся и не передаются в аргументах команд.

In [ ]:
%pip install -q -r requirements-notebook.txt comet_ml

In [ ]:
import json
import sys
from pathlib import Path

import torch
from omegaconf import OmegaConf

ROOT = Path.cwd()
INPUT_ROOT = Path("/kaggle/input/nlp-hw2-retrieval")
CHECKPOINT_PATH = INPUT_ROOT / "model_best.pth"

OUTPUT = ROOT / "artifacts/retrieval_val64"
RUN_NAME = "kaggle_retrieval_val64_lr1e5_20ep"
EPOCHS = 20
LR = 1e-5
TOP_K = 64
NUM_BEAMS = 4
TRAIN_BATCH_SIZE = 64
ACCUMULATION = 2
VAL_BATCH_SIZE = 32
RETRIEVAL_BATCH_SIZE = 128
RETRIEVAL_CHUNK_SIZE = 8192
EXECUTION_BATCH_SIZE = 64

In [ ]:
if not (ROOT / "train.py").is_file():
    raise FileNotFoundError("Перейди в папку скачанного репозитория NLP-HW2.")

# Только собственный checkpoint: он содержит конфиг OmegaConf.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False, mmap=True)
source_config = OmegaConf.create(checkpoint["config"])
print("Исходная эпоха:", checkpoint["epoch"])
del checkpoint

# Ссылки на большие данные вместо копирования в ограниченный working-диск.
relative_paths = [
    source_config.data.dataset_path,
    source_config.data.tokenizer_path,
    str(Path(source_config.datasets.train.indices_path).parent),
]
for relative in relative_paths:
    relative = Path(relative)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Ожидался путь внутри artifacts: {relative}")
    source = INPUT_ROOT / relative
    destination = ROOT / relative
    if not source.exists():
        raise FileNotFoundError(f"Нет файла или папки в Input: {source}")
    if destination.exists() or destination.is_symlink():
        if destination.resolve() != source.resolve():
            raise FileExistsError(f"Путь уже занят: {destination}")
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.symlink_to(source, target_is_directory=source.is_dir())

data_config = OmegaConf.to_container(source_config.data, resolve=True)
for key, default in (("code_format", "full"), ("add_imports", False), ("clean_questions", False)):
    data_config.setdefault(key, default)

experiment = OmegaConf.create({
    "defaults": ["tinysms_retrieval_val64", "_self_"],
    "model": OmegaConf.to_container(source_config.model, resolve=True),
    "data": data_config,
    "datasets": {"train": {"indices_path": str(OUTPUT / "train_indices.npy")}},
    "dataloader": {"batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": VAL_BATCH_SIZE},
    "optimizer": {"lr": LR},
    "trainer": {
        "from_pretrained": str(OUTPUT / "initial_model.pth"),
        "resume_from": None,
        "n_epochs": EPOCHS,
        "epoch_len": None,
        "accumulate_grad_batches": ACCUMULATION,
        "amp": {"enabled": True, "dtype": "auto"},
        "generation": {"num_beams": NUM_BEAMS},
    },
    "code_execution": {"sandbox": False, "batch_size": EXECUTION_BATCH_SIZE},
    "writer": {"run_name": RUN_NAME},
})
OmegaConf.save(experiment, ROOT / "src/configs/tinysms_retrieval_kaggle.yaml")
print("Конфиг готов; эффективный batch:", TRAIN_BATCH_SIZE * ACCUMULATION)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Включи GPU в настройках Kaggle Notebook.")
print("GPU:", torch.cuda.get_device_name(0))
print("AMP:", "BF16" if torch.cuda.is_bf16_supported(including_emulation=False) else "FP16")

Поиск просматривает весь отфильтрованный train. Для каждой validation-задачи берётся top-64 по cosine similarity encoder-векторов. Точные совпадения вопросов исключаются, найденные train-строки объединяются без повторения их индексов. Сохраняется фиксированная копия исходных весов.

Для повторного эксперимента выбери другую пустую папку `OUTPUT` и новое `RUN_NAME`. Следующая ячейка запускает реальный поиск на GPU.

In [ ]:
subprocess.run([
    sys.executable, "-u", "analyze_similarity.py",
    "--checkpoint", str(CHECKPOINT_PATH),
    "--query-split", "validation", "--top-k", str(TOP_K),
    "--exclude-identical-questions", "--save-train-subset",
    "--batch-size", str(RETRIEVAL_BATCH_SIZE),
    "--chunk-size", str(RETRIEVAL_CHUNK_SIZE),
    "--output", str(OUTPUT),
], cwd=ROOT, check=True)

summary = json.loads((OUTPUT / "summary.json").read_text())
print("Примеров в мини-датасете:", summary["subset_rows"])
print("Средний cosine ближайшего соседа:", summary["nearest_cosine"]["mean"])

Дообучение начинается с тех же фиксированных весов, что использовались при поиске, с новым оптимизатором и постоянным LR. Одна эпоха — полный проход мини-датасета.

До первого обновления и после каждой эпохи измеряются `val_loss`, `val_beam_pass_at_1` (лучший по score код из beam search, без голосования) и `val_greedy_pass_at_1`. Логи идут в Comet; сохраняются лучший checkpoint и 10 последних снимков. Поскольку вопросы validation определяют подбор train-примеров, это эксперимент по адаптации к validation, а не независимая оценка обобщения.

In [ ]:
subprocess.run([
    sys.executable, "-u", "train.py", "-cn=tinysms_retrieval_kaggle",
], cwd=ROOT, check=True)

In [ ]:
SAVE_DIR = ROOT / "saved" / RUN_NAME
print("Checkpoint и логи:", SAVE_DIR)
print("Лучший checkpoint:", SAVE_DIR / "model_best.pth")
print("Для возобновления вместо повторного запуска ячейки обучения:")
print(f"{sys.executable} train.py -cn=tinysms_retrieval_kaggle trainer.resume_from=checkpoint-last.pth")